Load the model from manual_run

In [1]:
import sys
sys.path.append('..') 

from pathlib import Path
import torch
from nam.models.nam import NAM
from nam.utils.config import load_config
from nam.data.data_utils import load_compas, preprocess, split
from scripts.train import build_model

CONFIG_PATH = r"C:\Users\maart\OneDrive\Documenten\Universiteit\Scriptie\python_repo\thesis-nam\configs\compas-scores-two-years_tuned.yaml"
CHECKPOINT  = Path(r"C:\Users\maart\OneDrive\Documenten\Universiteit\Scriptie\python_repo\thesis-nam\runs\manual_run\best.pt")

config = load_config(CONFIG_PATH)
df = load_compas(config.dataset_path)
X, y, _ = preprocess(df)
X_train, X_val, X_test, y_train, y_val, y_test = split(
    X, y, config.val_frac, config.test_frac, config.seed
)



Build the model with parameters

In [2]:
model = build_model(config, num_features=X_train.shape[1])
model.load_state_dict(torch.load(CHECKPOINT))
model.eval()



NAM(
  (feature_nns): ModuleList(
    (0-12): 13 x FeatureNN(
      (dropout): Dropout(p=0.1, inplace=False)
      (model): Sequential(
        (0): LinReLU()
        (1): Dropout(p=0.1, inplace=False)
        (2): Linear(in_features=64, out_features=64, bias=True)
        (3): ReLU()
        (4): Dropout(p=0.1, inplace=False)
        (5): Linear(in_features=64, out_features=32, bias=True)
        (6): ReLU()
        (7): Dropout(p=0.1, inplace=False)
        (8): Linear(in_features=32, out_features=1, bias=False)
      )
    )
  )
  (dropout_layer): Dropout(p=0.05, inplace=False)
)

Calculate the metric (without trainer)

In [3]:
from nam.training.metrics import auroc
from nam.data.dataset import NAMDataset
from torch.utils.data import DataLoader

test_loader = DataLoader(NAMDataset(X_test, y_test), batch_size=config.batch_size, shuffle=False)

all_predictions = []
all_targets = []

with torch.no_grad():
    for X_batch, y_batch, _ in test_loader:
        predictions, _ = model(X_batch)
        all_predictions.append(predictions)
        all_targets.append(y_batch)

predictions = torch.cat(all_predictions)
targets = torch.cat(all_targets)

metric = auroc(predictions, targets)
print(f"Test AUC: {metric:.4f}")

Test AUC: 0.7537
